# TurboQuant: How a Random Rotation Makes LLM Quantisation Near-Optimal

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/turboquant_random_rotations.ipynb)

Implement Google's TurboQuant algorithm from scratch in NumPy and understand why a random rotation is the key to near-optimal vector quantisation.

**Blog post:** [sesen.ai/blog/turboquant-vector-quantization-random-rotations](https://sesen.ai/blog/turboquant-vector-quantization-random-rotations)

**What you'll learn:**
- Why transformer activations have outlier coordinates that break naive quantisation
- How a random orthogonal rotation spreads energy evenly across all dimensions
- How to build an optimal scalar quantiser with the Lloyd-Max algorithm
- How the QJL residual correction produces unbiased inner product estimates

**Prerequisites:** Basic Python, NumPy, and familiarity with vectors and matrix multiplication.

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.integrate import quad
import matplotlib.pyplot as plt

## Part 1: The Outlier Problem

Transformer activations are not well-behaved. A handful of coordinates carry most of the magnitude while the rest are near zero. Let's create such a vector and see what happens when we quantise it naively.

In [ ]:
d = 128
rng = np.random.default_rng(0)

# Create a spiky vector: one huge coordinate, the rest near zero
x = rng.standard_normal(d) * 0.01
x[0] = 1.0
x = x / np.linalg.norm(x)  # normalise to unit length

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(d), np.abs(x), color='#f44336', alpha=0.8, width=1.0)
ax.axhline(1/np.sqrt(d), color='#666', linestyle='--', label=f'1/√d = {1/np.sqrt(d):.3f}')
ax.set_title('Spiky Vector: One Dominant Coordinate', fontweight='bold')
ax.set_xlabel('Coordinate index')
ax.set_ylabel('|value|')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Max coordinate: {np.max(np.abs(x)):.4f}")
print(f"Mean coordinate: {np.mean(np.abs(x)):.4f}")
print(f"Ratio: {np.max(np.abs(x)) / np.mean(np.abs(x)):.1f}x")

Naive quantisation maps each coordinate to the nearest point on a uniform grid. The dominant coordinate saturates (clips to the grid maximum) and the small coordinates round to zero. The result: huge information loss.

In [ ]:
# Naive uniform quantisation (3-bit = 8 levels)
grid = np.linspace(-1/np.sqrt(d), 1/np.sqrt(d), 8)
x_naive = grid[np.argmin(np.abs(x[:, None] - grid), axis=1)]
mse_naive = np.mean((x - x_naive) ** 2)

print(f"Naive quantisation MSE: {mse_naive:.6f}")
print(f"Cosine similarity:      {np.dot(x, x_naive) / (np.linalg.norm(x) * np.linalg.norm(x_naive)):.4f}")

## Part 2: The Random Rotation Fix

TurboQuant's key insight: randomly rotate the vector before quantising. This spreads the energy evenly across all coordinates, making every quantisation bit count.

We generate a **Haar-distributed** random orthogonal matrix via QR decomposition of a Gaussian matrix.

In [ ]:
def random_rotation_matrix(d, seed=42):
    """Haar-distributed random orthogonal matrix via QR decomposition."""
    rng = np.random.default_rng(seed)
    G = rng.standard_normal((d, d))
    Q, R = np.linalg.qr(G)
    # Fix sign ambiguity so Q is a proper rotation
    Q *= np.sign(np.diag(R))
    return Q

Pi = random_rotation_matrix(d, seed=42)

# Verify orthogonality and norm preservation
print(f"Orthogonal (Pi @ Pi.T = I): {np.allclose(Pi @ Pi.T, np.eye(d))}")
print(f"Norm preserved: {np.linalg.norm(x):.6f} -> {np.linalg.norm(Pi @ x):.6f}")

In [ ]:
# Rotate and visualise
y = Pi @ x

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(d), np.abs(x), color='#f44336', alpha=0.8, width=1.0)
ax1.axhline(1/np.sqrt(d), color='#666', linestyle='--', label=f'1/√d = {1/np.sqrt(d):.3f}')
ax1.set_title('Before Rotation', fontweight='bold')
ax1.set_xlabel('Coordinate index')
ax1.set_ylabel('|value|')
ax1.set_ylim(0, 1.05)
ax1.legend()

ax2.bar(range(d), np.abs(y), color='#4CAF50', alpha=0.8, width=1.0)
ax2.axhline(1/np.sqrt(d), color='#666', linestyle='--', label=f'1/√d = {1/np.sqrt(d):.3f}')
ax2.set_title('After Rotation', fontweight='bold')
ax2.set_xlabel('Coordinate index')
ax2.set_ylabel('|value|')
ax2.set_ylim(0, 1.05)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Before - max: {np.max(np.abs(x)):.4f}, std: {np.std(x):.4f}")
print(f"After  - max: {np.max(np.abs(y)):.4f}, std: {np.std(y):.4f}")

The spike is gone. After rotation, all coordinates have roughly the same magnitude, clustering around $1/\sqrt{d}$. This is the **concentration of measure** phenomenon on the unit sphere.

In high dimensions, each coordinate of a randomly rotated unit vector follows approximately $\mathcal{N}(0, 1/d)$.

## Part 3: The Lloyd-Max Optimal Scalar Quantiser

Now that every coordinate follows $\mathcal{N}(0, 1/d)$, we can design an optimal scalar quantiser for this distribution. The **Lloyd-Max** algorithm (continuous 1-D K-Means) finds centroids that minimise the expected MSE:

1. Initialise $2^b$ centroids uniformly
2. Set boundaries as midpoints between adjacent centroids
3. Update each centroid to $\mathbb{E}[X \mid X \in \text{partition}_i]$
4. Repeat until convergence

In [ ]:
def lloyd_max_codebook(d, bits, max_iter=200, tol=1e-10):
    """Optimal scalar quantiser for N(0, 1/d) via Lloyd-Max iteration."""
    n_levels = 2 ** bits
    sigma = 1.0 / np.sqrt(d)
    pdf = lambda x: norm.pdf(x, 0, sigma)

    # Initialise centroids uniformly across [-3.5*sigma, 3.5*sigma]
    lo, hi = -3.5 * sigma, 3.5 * sigma
    centroids = np.array([lo + (hi - lo) * (i + 0.5) / n_levels
                          for i in range(n_levels)])

    for iteration in range(max_iter):
        boundaries = (centroids[:-1] + centroids[1:]) / 2.0
        edges = np.concatenate([[-np.inf], boundaries, [np.inf]])
        new_centroids = np.empty(n_levels)
        for i in range(n_levels):
            num, _ = quad(lambda x: x * pdf(x), edges[i], edges[i + 1])
            den, _ = quad(pdf, edges[i], edges[i + 1])
            new_centroids[i] = num / den if den > 1e-15 else centroids[i]
        if np.max(np.abs(new_centroids - centroids)) < tol:
            break
        centroids = new_centroids

    boundaries = (centroids[:-1] + centroids[1:]) / 2.0
    return centroids, boundaries

def quantize_scalar(x, centroids):
    """Map each scalar to the index of its nearest centroid."""
    return np.argmin(np.abs(x[:, None] - centroids), axis=1)

In [ ]:
# Compute codebooks for different bit-widths
for bits in [1, 2, 3, 4]:
    centroids, boundaries = lloyd_max_codebook(d, bits)
    print(f"{bits}-bit centroids: {np.round(centroids, 5)}")

Notice how the centroids are denser near zero (where the PDF peaks) and sparser in the tails. A uniform grid wastes levels on low-probability regions.

In [ ]:
# Visualise: Lloyd-Max centroids vs the distribution
centroids_3bit, boundaries_3bit = lloyd_max_codebook(d, bits=3)
sigma = 1.0 / np.sqrt(d)
xs = np.linspace(-4*sigma, 4*sigma, 300)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(xs, norm.pdf(xs, 0, sigma), 'b-', linewidth=2, label='N(0, 1/d)')
for b in boundaries_3bit:
    ax.axvline(b, color='#999', linestyle='--', linewidth=0.8)
for c in centroids_3bit:
    ax.axvline(c, color='#f44336', linewidth=2, alpha=0.7)
ax.scatter(centroids_3bit, norm.pdf(centroids_3bit, 0, sigma),
           color='#f44336', s=80, zorder=5, label='Centroids')
ax.set_title('3-bit Lloyd-Max Codebook for N(0, 1/128)', fontweight='bold')
ax.set_xlabel('Value')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Putting It Together — Full TurboQuant MSE

The complete pipeline: rotate, quantise each coordinate with Lloyd-Max, dequantise, unrotate.

In [ ]:
def turboquant_mse(x, bits=3, seed=42):
    """
    TurboQuant MSE quantise-dequantise cycle.
    x: (d,) unit-norm vector
    Returns: reconstructed vector, MSE
    """
    d = len(x)
    Pi = random_rotation_matrix(d, seed)
    centroids, _ = lloyd_max_codebook(d, bits)

    # Rotate -> quantise each coordinate -> dequantise -> unrotate
    y = Pi @ x
    indices = quantize_scalar(y, centroids)
    y_hat = centroids[indices]
    x_hat = Pi.T @ y_hat

    mse = np.mean((x - x_hat) ** 2)
    return x_hat, mse

In [ ]:
# Compare naive vs TurboQuant on the spiky vector
x_hat, mse_turbo = turboquant_mse(x, bits=3)

print(f"Naive quantisation MSE:     {mse_naive:.6f}")
print(f"TurboQuant MSE:             {mse_turbo:.6f}")
print(f"Improvement:                {mse_naive / mse_turbo:.1f}x")
print()
print(f"Naive cosine similarity:    {np.dot(x, x_naive) / (np.linalg.norm(x) * np.linalg.norm(x_naive)):.4f}")
print(f"TurboQuant cosine sim:      {np.dot(x, x_hat) / (np.linalg.norm(x) * np.linalg.norm(x_hat)):.4f}")

Over 30x improvement on a spiky vector. Let's see how the improvement varies across bit-widths and compare against the information-theoretic lower bound.

In [ ]:
bits_range = [1, 2, 3, 4]
mse_naive_list, mse_turbo_list, mse_lower_bound = [], [], []

for bits in bits_range:
    # TurboQuant
    _, mse_t = turboquant_mse(x, bits=bits)
    mse_turbo_list.append(mse_t)

    # Naive uniform
    n_levels = 2 ** bits
    grid = np.linspace(-1.0, 1.0, n_levels)
    x_n = grid[np.argmin(np.abs(x[:, None] - grid), axis=1)]
    mse_naive_list.append(np.mean((x - x_n) ** 2))

    # Info-theoretic lower bound for N(0, 1/d)
    mse_lower_bound.append((1.0 / d) * 2**(-2 * bits))

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(bits_range, mse_naive_list, 'r-o', linewidth=2, markersize=8, label='Naive Uniform')
ax.semilogy(bits_range, mse_turbo_list, 'g-s', linewidth=2, markersize=8, label='TurboQuant (Lloyd-Max)')
ax.semilogy(bits_range, mse_lower_bound, 'k--^', linewidth=1.5, markersize=7,
            label='Info-Theoretic Lower Bound')
ax.set_xlabel('Bits per coordinate', fontsize=12)
ax.set_ylabel('MSE (log scale)', fontsize=12)
ax.set_title('Quantisation Distortion: TurboQuant vs Naive', fontweight='bold')
ax.set_xticks(bits_range)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

TurboQuant tracks close to the information-theoretic lower bound across all bit-widths, while naive quantisation is orders of magnitude worse on outlier-heavy vectors.

## Part 5: Does Rotation Help on "Normal" Vectors?

What if the vector doesn't have outliers? Let's test on a random unit vector with no dominant coordinate.

In [ ]:
# Well-behaved random vector (no outliers)
x_normal = rng.standard_normal(d)
x_normal = x_normal / np.linalg.norm(x_normal)

# Naive
grid_3 = np.linspace(-1/np.sqrt(d), 1/np.sqrt(d), 8)
x_n_normal = grid_3[np.argmin(np.abs(x_normal[:, None] - grid_3), axis=1)]
mse_naive_normal = np.mean((x_normal - x_n_normal) ** 2)

# TurboQuant
_, mse_turbo_normal = turboquant_mse(x_normal, bits=3)

print(f"Well-behaved vector (no outliers):")
print(f"  Naive MSE:      {mse_naive_normal:.6f}")
print(f"  TurboQuant MSE: {mse_turbo_normal:.6f}")
print(f"  Ratio:          {mse_naive_normal / mse_turbo_normal:.2f}x")
print()
print("The rotation still helps (Lloyd-Max is better than uniform),")
print("but the gap is much smaller because the vector was already well-spread.")

The rotation never hurts. On well-behaved vectors the improvement is modest; on spiky vectors it is dramatic. This is the beauty of the data-oblivious design.

## Part 6: Unbiased Inner Products via QJL

In attention, we compute $\langle \mathbf{q}, \mathbf{k} \rangle$ between queries and cached keys. MSE-quantised keys produce **biased** inner products. TurboQuant fixes this with a second stage: the **Quantized Johnson-Lindenstrauss (QJL)** transform on the residual.

Of the $b$-bit budget, $b-1$ bits go to Lloyd-Max MSE, and 1 bit stores the sign of a random projection of the residual:

$$\langle \mathbf{q}, \mathbf{k} \rangle \approx \langle \mathbf{q}, \hat{\mathbf{k}}_{\text{MSE}} \rangle + \|\mathbf{r}\| \cdot \sqrt{\frac{\pi}{2m}} \cdot \langle S\mathbf{q},\, \text{sign}(S\mathbf{r}) \rangle$$

In [ ]:
def turboquant_inner_product(q, k, bits=3, seed=42):
    """
    Unbiased inner product estimation using TurboQuant Stage 1 + 2.
    q, k: (d,) unit-norm vectors
    Returns: estimated <q, k>
    """
    d = len(k)
    mse_bits = bits - 1  # reserve 1 bit for QJL

    # Stage 1: MSE quantise the key
    Pi = random_rotation_matrix(d, seed)
    centroids, _ = lloyd_max_codebook(d, mse_bits)
    y = Pi @ k
    indices = quantize_scalar(y, centroids)
    y_hat = centroids[indices]
    k_mse = Pi.T @ y_hat

    # Compute residual
    r = k - k_mse
    r_norm = np.linalg.norm(r)

    # Stage 2: QJL on residual
    rng = np.random.default_rng(seed + 1)
    S = rng.standard_normal((d, d))  # projection matrix
    qjl_signs = np.sign(S @ r)
    qjl_signs[qjl_signs == 0] = 1.0

    # Unbiased inner product estimate
    term1 = q @ k_mse
    term2 = r_norm * np.sqrt(np.pi / 2) / d * (S @ q) @ qjl_signs
    return term1 + term2

In [ ]:
# Test inner product estimation over many random vector pairs
# Precompute codebook and rotation once (these are expensive)
d_test = 128
n_trials = 50

Pi_test = random_rotation_matrix(d_test, seed=999)
centroids_mse, _ = lloyd_max_codebook(d_test, bits=2)  # b-1 = 2 bits for MSE stage
centroids_full, _ = lloyd_max_codebook(d_test, bits=3)  # full 3 bits for MSE-only

true_ips, mse_ips, turbo_ips = [], [], []

for trial in range(n_trials):
    rng_t = np.random.default_rng(trial)
    q = rng_t.standard_normal(d_test); q /= np.linalg.norm(q)
    k = rng_t.standard_normal(d_test)
    k[0] = 5.0  # add outlier
    k /= np.linalg.norm(k)

    # True inner product
    true_ips.append(q @ k)

    # MSE-only (3 bits, reusing precomputed rotation + codebook)
    y_k = Pi_test @ k
    idx_full = quantize_scalar(y_k, centroids_full)
    k_mse_full = Pi_test.T @ centroids_full[idx_full]
    mse_ips.append(q @ k_mse_full)

    # TurboQuant (2-bit MSE + 1-bit QJL)
    idx_mse = quantize_scalar(y_k, centroids_mse)
    k_mse = Pi_test.T @ centroids_mse[idx_mse]
    r = k - k_mse
    r_norm = np.linalg.norm(r)
    S = np.random.default_rng(trial + 1000).standard_normal((d_test, d_test))
    qjl_signs = np.sign(S @ r)
    qjl_signs[qjl_signs == 0] = 1.0
    term1 = q @ k_mse
    term2 = r_norm * np.sqrt(np.pi / 2) / d_test * (S @ q) @ qjl_signs
    turbo_ips.append(term1 + term2)

true_ips = np.array(true_ips)
turbo_ips = np.array(turbo_ips)
mse_ips = np.array(mse_ips)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(true_ips, mse_ips, alpha=0.5, s=20, label='MSE only')
lims = [min(true_ips.min(), mse_ips.min()) - 0.05, max(true_ips.max(), mse_ips.max()) + 0.05]
ax1.plot(lims, lims, 'k--', linewidth=1)
ax1.set_xlabel('True inner product')
ax1.set_ylabel('Estimated inner product')
ax1.set_title('MSE-only Reconstruction', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.scatter(true_ips, turbo_ips, alpha=0.5, s=20, color='green', label='MSE + QJL')
lims2 = [min(true_ips.min(), turbo_ips.min()) - 0.05, max(true_ips.max(), turbo_ips.max()) + 0.05]
ax2.plot(lims2, lims2, 'k--', linewidth=1)
ax2.set_xlabel('True inner product')
ax2.set_ylabel('Estimated inner product')
ax2.set_title('TurboQuant (MSE + QJL)', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"MSE-only   - Mean error: {np.mean(np.abs(true_ips - mse_ips)):.6f}, "
      f"Correlation: {np.corrcoef(true_ips, mse_ips)[0,1]:.4f}")
print(f"MSE + QJL  - Mean error: {np.mean(np.abs(true_ips - turbo_ips)):.6f}, "
      f"Correlation: {np.corrcoef(true_ips, turbo_ips)[0,1]:.4f}")

The QJL correction brings the inner product estimates closer to the true values. This matters for attention: biased inner products mean the model attends to the wrong tokens.

## Part 7: Memory Savings

Let's calculate the actual compression ratio for a realistic KV cache scenario.

In [ ]:
# Realistic KV cache parameters
seq_len = 8192       # 8K context
n_layers = 32        # typical for 7-8B models
n_heads = 32
d_head = 128         # head dimension

# FP16 baseline
fp16_bits = 16
kv_fp16_bits = 2 * seq_len * n_layers * n_heads * d_head * fp16_bits  # keys + values
kv_fp16_gb = kv_fp16_bits / 8 / 1e9

# TurboQuant at 3 bits
turbo_bits_per_coord = 3
# Keys: (b-1) bits MSE + 1 bit QJL + 16 bits for residual norm per vector
key_bits = seq_len * n_layers * n_heads * (d_head * turbo_bits_per_coord + 16)
# Values: b bits MSE (no QJL needed)
val_bits = seq_len * n_layers * n_heads * d_head * turbo_bits_per_coord
kv_turbo_bits = key_bits + val_bits
kv_turbo_gb = kv_turbo_bits / 8 / 1e9

print(f"KV Cache Memory (8K context, 32-layer model):")
print(f"  FP16:       {kv_fp16_gb:.2f} GB")
print(f"  TurboQuant: {kv_turbo_gb:.2f} GB  (3 bits/coord)")
print(f"  Compression: {kv_fp16_gb / kv_turbo_gb:.1f}x")
print()
print(f"At 128K context, FP16 would need {kv_fp16_gb * 16:.1f} GB!")
print(f"TurboQuant brings it down to {kv_turbo_gb * 16:.1f} GB.")

## Exercises

1. **Vary the dimension** — Run TurboQuant on vectors with d = 32, 64, 128, 256, 512. How does the MSE improvement (vs naive) change with dimension? Why?

2. **Multiple outliers** — Instead of one dominant coordinate, set coordinates 0-4 to large values. Does TurboQuant still help as much?

3. **Hadamard rotation** — Replace the random rotation with the Walsh-Hadamard transform (available in `scipy.linalg.hadamard`). Compare MSE and speed against the QR-based rotation. At what dimensions does the $O(d \log d)$ vs $O(d^2)$ difference matter?

4. **Verify unbiasedness** — For the QJL inner product estimator, compute the mean estimate over 1000 random seeds for a fixed (q, k) pair. Does it converge to the true inner product?

5. **Simulate attention** — Create a batch of 100 key vectors with outliers, quantise them with TurboQuant, then compute attention weights against a query. Compare the softmax output with the full-precision version.

## References

- Zandieh, A., Daliri, M., Hadian, M., & Mirrokni, V. (2025). TurboQuant: Online Vector Quantization with Near-optimal Distortion Rate. *arXiv:2504.19874*. https://arxiv.org/abs/2504.19874
- Chee, J. et al. (2024). QuIP: 2-Bit Quantization of Large Language Models With Guarantees. *arXiv:2307.13304*.
- Sun, C. et al. (2024). Massive Activations in Large Language Models. *arXiv:2402.17762*.
- Lloyd, S. (1982). Least Squares Quantization in PCM. *IEEE Transactions on Information Theory*, 28(2), 129-137.